# M3 · Retrieval avanzado y orquestación de la normalización terminológica

Este notebook ejecuta y evalúa el retrieval del sistema RAG sobre guías clínicas. La lógica vive en
módulos `.py`; aquí solo se llaman en orden:

1. Pruebas unitarias.
2. Construcción de un índice Chroma de prueba.
3. Prueba de humo de las técnicas de búsqueda.
4. Experimento controlado: todos los sistemas sobre las mismas consultas.
5. Análisis por tipo de consulta y decisiones de la normalización.
6. Comparación con otro reranker (opcional).
7. Integración con la generación de respuestas.

La implementación parte del laboratorio S08 (BM25, fusión RRF con k = 60 y reranker cross-encoder
mMARCO) y lo adapta al corpus del proyecto. Las rutas, los modelos y los parámetros están en
`M3/retrieval/config_retrieval.yaml`; para usar el corpus definitivo basta con cambiar `corpus: mock`
por `corpus: real` en ese archivo.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

RAIZ = '/content/drive/MyDrive/TopicosIA/Proyecto-Salud'   # carpeta que contiene data/ y M3/
%cd {RAIZ}

Mounted at /content/drive
/content/drive/MyDrive/TopicosIA/Proyecto-Salud


In [2]:
%pip install -q sentence-transformers chromadb rank_bm25 groq python-dotenv PyYAML

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 115.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6

In [3]:
import os, shutil, sys

if not os.path.exists('corpus_utils.py'):
    shutil.copy('M3/retrieval/_corpus_utils_referencia.py', 'corpus_utils.py')
    print('Se usa la versión de referencia de corpus_utils.py.')
for ruta in ['.', 'M3/retrieval', 'M3/generacion']:
    if ruta not in sys.path:
        sys.path.insert(0, ruta)

## 1 · Pruebas unitarias

Verifican la lógica con modelos simulados, sin descargar nada.

In [4]:
!cd M3/retrieval && python -m unittest test_retrieval
!cd M3/generacion && python -m unittest test_generacion

...............................
----------------------------------------------------------------------
Ran 31 tests in 1.696s

OK
....    [orquestacion] falló el retrieval ('neumonia'): indice caido
    [orquestacion] falló el retrieval ('ENFERMEDAD CONTROLADA'): indice caido
....
----------------------------------------------------------------------
Ran 8 tests in 0.002s

OK


## 2 · Configuración e índice de prueba

La configuración se lee de `config_retrieval.yaml`. Con `corpus: mock`, el índice se construye a
partir de un corpus de prueba de 13 guías con tres secciones cada una (descripción clínica,
diagnóstico y tratamiento), usando el mismo código de ingesta del corpus real. El índice se guarda
en el disco local de Colab, porque Chroma usa SQLite y puede fallar sobre el disco montado de Drive.

In [5]:
from config_retrieval import cargar_config, configurar_pipeline, indice_desde_config, parametros_orquestacion, rutas

cfg = cargar_config('M3/retrieval/config_retrieval.yaml')
R = rutas(cfg)
print('Corpus:', cfg['corpus'], '| índice:', R['chroma_dir'], '| resultados:', R['salida'])

if cfg['corpus'] == 'mock':
    !python M3/retrieval/construir_indice_mock.py

Corpus: mock | índice: /content/chroma_mock | resultados: M3/retrieval/outputs_mock
modules.json: 100% 387/387 [00:00<00:00, 2.11MB/s]
README.md: 100% 179k/179k [00:00<00:00, 174MB/s]
sentence_bert_config.json: 100% 57.0/57.0 [00:00<00:00, 278kB/s]
config.json: 100% 694/694 [00:00<00:00, 4.42MB/s]

model.safetensors: downloading bytes:  15% 164M/1.11G [00:01<00:04, 192MB/s, 12.3MB/s  ]
model.safetensors: downloading bytes:  19% 208M/1.11G [00:02<00:05, 169MB/s, 18.1MB/s  ]
model.safetensors: downloading bytes:  29% 323M/1.11G [00:02<00:03, 236MB/s, 26.4MB/s  ]
model.safetensors: downloading bytes:  65% 722M/1.11G [00:03<00:01, 322MB/s, 58.0MB/s  ]
model.safetensors: downloading bytes: 100% 741M/741M [00:08<00:00, 84.4MB/s, 61.9MB/s  ]
model.safetensors: reconstructing file: 100% 1.11G/1.11G [00:08<00:00, 127MB/s, 84.4MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 4278.08it/s]
tokenizer_config.json: 100% 418/418 [00:00<00:00, 2.27MB/s]

sentencepiece.bpe.model: downloading bytes:  

## 3 · Prueba de humo

In [6]:
import collections
from retrieval import info_indice

indice = configurar_pipeline(cfg)
info = info_indice(indice)
print(f"Índice: {info['n_chunks']} chunks, {info['n_documentos']} guías, huella {info['huella_sha256']}")
print('Secciones:', dict(collections.Counter(zip(indice.seccion_tipo, indice.seccion_origen))))

MODOS_HUMO = {'densa': 'Denso (ingenuo)', 'rerank': 'Híbrido + reranker',
              'intencion_sin_rerank': 'Tratamiento, sin reranker', 'intencion': 'Tratamiento + reranker',
              'intencion_seccion': 'Tratamiento + reranker + sección'}
for q in ['hipertensión arterial', 'HTA', 'paludismo', 'diabetes en el embarazo', 'EPOC']:
    print(f'\n{q}')
    for modo, nombre in MODOS_HUMO.items():
        top = indice.buscar(q, 3, modo)
        print(f'  {nombre:34}', [(f['chunk_id'].replace('mock_', ''), f['seccion_tipo'], round(f['score'], 3))
                                   for f in top])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Índice: 39 chunks, 13 guías, huella 6d80190deae48d08
Secciones: {('otra', 'titulo'): 24, ('tratamiento', 'titulo'): 15}

hipertensión arterial
  Denso (ingenuo)                    [('hipertension_0', 'otra', 0.915), ('hipertension_1', 'tratamiento', 0.898), ('hipertension_2', 'tratamiento', 0.88)]
  Híbrido + reranker                 [('hipertension_0', 'otra', 0.964), ('hipertension_1', 'tratamiento', 0.447), ('hipertension_2', 'tratamiento', 0.393)]
  Tratamiento, sin reranker          [('hipertension_1', 'tratamiento', 0.098), ('hipertension_2', 'tratamiento', 0.097), ('hipertension_0', 'otra', 0.096)]
  Tratamiento + reranker             [('hipertension_2', 'tratamiento', 0.994), ('hipertension_1', 'tratamiento', 0.981), ('hipertension_0', 'otra', 0.045)]
  Tratamiento + reranker + sección   [('hipertension_2', 'tratamiento', 0.994), ('hipertension_1', 'tratamiento', 0.981), ('diabetes_1', 'tratamiento', 0.025)]

HTA
  Denso (ingenuo)                    [('hipertension_0', 'otra', 

Cada resultado muestra el chunk, el tipo de sección y el score. Conviene revisar tres cosas: que las
técnicas con consultas de tratamiento pongan primero un chunk de sección `tratamiento`, que las
consultas fuera del corpus (EPOC) reciban scores bajos del reranker y cuántos chunks se clasificaron
por título y cuántos por similitud. En el corpus real, los clasificados por similitud deben revisarse
a mano en una muestra.

## 4 · Experimento controlado

| Sistema | Qué agrega |
|---|---|
| Denso (RAG ingenuo) | Línea base de S07: similitud de embeddings |
| Híbrido BM25 + denso | Búsqueda léxica fusionada con RRF |
| Híbrido + reranker | Cross-encoder sobre los 30 mejores candidatos |
| Híbrido + reranker con pregunta de tratamiento | El reranker recibe «¿Cuál es el tratamiento de…?» |
| Consultas de tratamiento, sin reranker | Tres reformulaciones con intención fusionadas con RRF |
| Consultas de tratamiento + reranker | Las reformulaciones más el reranker con la pregunta |
| … + filtro de sección | El reranker solo ordena chunks de secciones de tratamiento |
| Sistema completo | Normalización terminológica y compuerta de evidencia sobre `orquestacion.sistema_base` del YAML |
| Sistema completo + regla de siglas | Normaliza siempre las siglas |

Las dos filas de la mitad (reranker con pregunta y consultas sin reranker) separan qué aporta cada
pieza de la query transformation. Las métricas «Tratamiento» exigen que llegue el chunk de
tratamiento, no solo la guía correcta.

Los umbrales del sistema completo se calibran con validación cruzada: cada consulta se evalúa con
umbrales calibrados en los otros pliegues. En el corpus de prueba, las etiquetas de sección provienen
de los mismos títulos que usa el filtro de sección, así que ese sistema tiene ventaja por construcción.

In [7]:
import pandas as pd
from experimento_s08 import NOMBRE, TIPOS_CONSULTA, cargar_consultas, correr_desde_config, correr_experimento

detalle, resumen, calibracion = correr_desde_config(cfg, indice)

tabla = pd.DataFrame(resumen).drop(columns='Clave').set_index('Sistema')
display(tabla.T)

Sistema,Denso (RAG ingenuo),Híbrido BM25 + denso,Híbrido + reranker,Híbrido + reranker con pregunta de tratamiento,"Consultas de tratamiento, sin reranker",Consultas de tratamiento + reranker,Consultas de tratamiento + reranker + filtro de sección,Sistema completo (normalización + compuerta),Sistema completo + regla de siglas
Consultas con guía en el corpus,27.000,27.000,27.000,27.000,27.000,27.000,27.000,27.000,27.000
Guía correcta en 1.er lugar,0.926,0.926,0.778,0.815,0.667,0.778,0.778,0.889,0.963
Guía correcta en el top-5,1.000,1.000,0.889,0.852,0.963,0.852,0.852,0.926,0.963
MRR de la guía,0.952,0.952,0.827,0.833,0.769,0.802,0.802,0.901,0.963
Guías relevantes en el top-5,1.000,1.000,0.889,0.852,0.963,0.852,0.852,0.926,0.963
Precisión de guía en el top-5,0.585,0.585,0.459,0.474,0.459,0.467,0.230,0.871,0.942
Tratamiento en 1.er lugar,0.000,0.074,0.037,0.815,0.593,0.778,0.778,0.889,0.963
Tratamiento en el top-5,0.963,0.963,0.741,0.852,0.963,0.852,0.852,0.926,0.963
MRR del tratamiento,0.372,0.409,0.308,0.827,0.725,0.799,0.802,0.901,0.963
Precisión de tratamiento en el top-5,0.244,0.244,0.178,0.230,0.244,0.230,0.230,0.765,0.769


In [8]:
cols = {'umbral': 'Umbral para normalizar', 'umbral_evidencia': 'Umbral de evidencia',
        'hay_zona_dudosa': 'Existe zona dudosa', 'evidencia_conservada': 'Evidencia correcta conservada'}
umbrales = pd.DataFrame([{'Calibración': f"Pliegue {p['pliegue']} (sin sus consultas)", **p} for p in calibracion['pliegues']]
                        + [{'Calibración': 'Final (todas las consultas)', **calibracion['final']}])
display(umbrales.set_index('Calibración')[list(cols)].rename(columns=cols))
print('Validación:', calibracion['validacion'])

,Umbral para normalizar,Umbral de evidencia,Existe zona dudosa,Evidencia correcta conservada
Calibración,,,,
Pliegue 0 (sin sus consultas),0.753524,0.553563,True,1.000
Pliegue 1 (sin sus consultas),0.334961,0.027144,True,0.923
Pliegue 2 (sin sus consultas),0.753524,0.553563,True,0.846
Final (todas las consultas),0.753524,0.553563,True,0.895


Validación: cruzada, 3 pliegues estratificados por tipo de consulta


Los umbrales finales son los que se configuran en el pipeline. Si varían mucho entre pliegues, la
calibración todavía depende de pocas consultas. «Existe zona dudosa» indica si el umbral de evidencia
quedó por debajo del de normalización; si no, el sistema solo distingue entre confiable y sin evidencia.

## 5 · Análisis por tipo de consulta

In [9]:
df = pd.DataFrame(detalle)
orden = [NOMBRE[f['Clave']] for f in resumen]
dentro = df[~df.fuera_de_corpus]

def por_tipo(campo):
    t = dentro.pivot_table(index='tipo', columns='sistema', values=campo, aggfunc='mean')
    return t[[c for c in orden if c in t.columns]].rename(index=TIPOS_CONSULTA).rename_axis(index='Tipo de consulta',
                                                                                           columns=None).round(2).T

print('Tratamiento en 1.er lugar, por tipo de consulta')
display(por_tipo('hit1_sec'))
print('MRR de la guía, por tipo de consulta')
display(por_tipo('rr'))

Tratamiento en 1.er lugar, por tipo de consulta


Tipo de consulta,Guías parecidas,Errata o sin tildes,Nombre exacto,Hiperespecífica,Sigla,Sinónimo
Denso (RAG ingenuo),0.00,0.0,0.00,0.0,0.00,0.00
Híbrido BM25 + denso,0.00,0.0,0.17,0.0,0.00,0.17
Híbrido + reranker,0.00,0.0,0.00,0.0,0.17,0.00
Híbrido + reranker con pregunta de tratamiento,1.00,1.0,1.00,1.0,0.67,0.50
"Consultas de tratamiento, sin reranker",0.67,1.0,1.00,1.0,0.17,0.17
Consultas de tratamiento + reranker,1.00,1.0,1.00,1.0,0.50,0.50
Consultas de tratamiento + reranker + filtro de sección,1.00,1.0,1.00,1.0,0.50,0.50
Sistema completo (normalización + compuerta),1.00,1.0,1.00,1.0,0.67,0.83
Sistema completo + regla de siglas,1.00,1.0,1.00,1.0,1.00,0.83


MRR de la guía, por tipo de consulta


Tipo de consulta,Guías parecidas,Errata o sin tildes,Nombre exacto,Hiperespecífica,Sigla,Sinónimo
Denso (RAG ingenuo),1.0,1.0,1.0,1.0,1.00,0.78
Híbrido BM25 + denso,1.0,1.0,1.0,1.0,1.00,0.78
Híbrido + reranker,1.0,1.0,1.0,1.0,0.64,0.58
Híbrido + reranker con pregunta de tratamiento,1.0,1.0,1.0,1.0,0.67,0.58
"Consultas de tratamiento, sin reranker",1.0,1.0,1.0,1.0,0.43,0.53
Consultas de tratamiento + reranker,1.0,1.0,1.0,1.0,0.56,0.56
Consultas de tratamiento + reranker + filtro de sección,1.0,1.0,1.0,1.0,0.56,0.56
Sistema completo (normalización + compuerta),1.0,1.0,1.0,1.0,0.72,0.83
Sistema completo + regla de siglas,1.0,1.0,1.0,1.0,1.00,0.83


In [10]:
piezas = ['reranker', 'reranker_pregunta', 'intencion_sin_reranker', 'intencion']
aporte = tabla.loc[[NOMBRE[p] for p in piezas], ['Tratamiento en 1.er lugar', 'MRR del tratamiento',
                                                   'Latencia (ms por consulta)']]
display(aporte)

,Tratamiento en 1.er lugar,MRR del tratamiento,Latencia (ms por consulta)
Sistema,,,
Híbrido + reranker,0.037,0.308,66.7
Híbrido + reranker con pregunta de tratamiento,0.815,0.827,83.8
"Consultas de tratamiento, sin reranker",0.593,0.725,62.4
Consultas de tratamiento + reranker,0.778,0.799,98.4


Esta tabla separa las dos piezas de la query transformation. Si «Consultas de tratamiento, sin
reranker» ya alcanza a «Consultas de tratamiento + reranker», la mejora viene de las reformulaciones y
el reranker no se gana su lugar; si queda por debajo, el reranker sí aporta. «Híbrido + reranker con
pregunta de tratamiento» mide el efecto de la pregunta sin las reformulaciones.

In [11]:
decisiones = df[df.clave == 'completo_siglas'][['consulta', 'tipo', 'tool', 'query_final', 'hit1_sec', 'rechazo',
                                                'tool_reason']]
decisiones = decisiones.assign(tipo=decisiones.tipo.map(TIPOS_CONSULTA)).rename(columns={
    'consulta': 'Consulta', 'tipo': 'Tipo de consulta', 'tool': 'Usó la normalización', 'query_final': 'Consulta final',
    'hit1_sec': 'Tratamiento en 1.er lugar', 'rechazo': 'Sin fragmentos entregados', 'tool_reason': 'Motivo'})
display(decisiones.reset_index(drop=True))

,Consulta,Tipo de consulta,Usó la normalización,Consulta final,Tratamiento en 1.er lugar,Sin fragmentos entregados,Motivo
0,diabetes mellitus tipo 2,Nombre exacto,0.0,diabetes mellitus tipo 2,1.0,0,
1,hipertensión arterial,Nombre exacto,0.0,hipertensión arterial,1.0,0,
2,asma,Nombre exacto,0.0,asma,1.0,0,
3,migraña,Nombre exacto,0.0,migraña,1.0,0,
4,osteoporosis,Nombre exacto,0.0,osteoporosis,1.0,0,
5,malaria,Nombre exacto,0.0,malaria,1.0,0,
6,DM2,Sigla,1.0,diabetes mellitus tipo 2,1.0,0,entidad con forma de sigla (score=0.999); norm...
7,HTA,Sigla,1.0,hipertensión arterial,1.0,0,entidad con forma de sigla (score=0.880); norm...
8,ITU,Sigla,1.0,infección del tracto urinario,1.0,0,entidad con forma de sigla (score=0.534); norm...
9,AR,Sigla,1.0,artritis reumatoide,1.0,0,entidad con forma de sigla (score=0.902); norm...


## 6 · Otro reranker (opcional)

El reranker alternativo del YAML (`BAAI/bge-reranker-v2-m3`) es multilingüe y más grande. Conviene ejecutarlo con GPU
(Entorno de ejecución › Cambiar tipo de entorno de ejecución › T4).

In [12]:
indice_bge = indice_desde_config(cfg, modelo_reranker=cfg['modelos']['reranker_alternativo'])
_, resumen_bge, _ = correr_experimento(indice_bge, cargar_consultas(R['consultas']), k=cfg['retrieval']['k'],
                                      claves=('ingenuo', 'reranker', 'intencion'))
comparacion = pd.concat([
    tabla.loc[[NOMBRE['reranker'], NOMBRE['intencion']]].rename(index=lambda s: s + ' (mMARCO)'),
    pd.DataFrame(resumen_bge).drop(columns='Clave').set_index('Sistema')
      .loc[[NOMBRE['reranker'], NOMBRE['intencion']]].rename(index=lambda s: s + ' (bge-m3)'),
])
display(comparacion[['Guía correcta en 1.er lugar', 'Tratamiento en 1.er lugar', 'MRR del tratamiento',
                     'Latencia (ms por consulta)']])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

,Guía correcta en 1.er lugar,Tratamiento en 1.er lugar,MRR del tratamiento,Latencia (ms por consulta)
Sistema,,,,
Híbrido + reranker (mMARCO),0.778,0.037,0.308,66.7
Consultas de tratamiento + reranker (mMARCO),0.778,0.778,0.799,98.4
Híbrido + reranker (bge-m3),0.926,0.111,0.470,595.0
Consultas de tratamiento + reranker (bge-m3),0.926,0.926,0.948,685.3


## 7 · Integración con la generación

Se ejecuta la cadena completa para tres entidades: orquestación, retrieval y generación de la
respuesta. La generación recibe como parámetro la función que llama al LLM; aquí se usa una función
simulada que devuelve un texto fijo, para verificar el recorrido de los datos sin consumir la cuota de
la API. Para obtener respuestas reales se reemplaza por `GroqGenerator()` del módulo `generacion`.

Con el corpus real, el normalizador simulado se reemplaza por `normalizacion.normalizar_entidad`.

In [13]:
import json
from retrieval import pregunta_intencion, retrieve_advanced
from orquestacion import resolver_query
from generacion import generar_respuesta
from mocks import crear_normalizador_mock

parametros = parametros_orquestacion(cfg)       # umbrales finales de la calibración
mapa = json.load(open(R['mapa_normalizacion'], encoding='utf-8'))
normalizar = crear_normalizador_mock({k.lower(): v for k, v in mapa.items()})

def generador_simulado(system, user):
    return 'Respuesta simulada con la fuente citada.'

for entidad in ['neumonía adquirida en la comunidad', 'HTA', 'amiloidosis']:
    q = resolver_query(entidad, retrieve_fn=retrieve_advanced, normalizar_fn=normalizar, **parametros)
    r = generar_respuesta(q['query_final'], q['fragments'], generar_fn=generador_simulado,
                          pregunta=pregunta_intencion(q['query_final']))
    print(f"\n{entidad}")
    print('  normalización:', 'sí' if q['tool_invoked'] else 'no', '|', q['tool_reason'] or 'score suficiente')
    print('  fragmentos:', [(f['chunk_id'], f['seccion_tipo'], f['score']) for f in q['fragments']])
    print('  respuesta:', r['answer'], '| sin evidencia' if r['fallback_used'] else '')


neumonía adquirida en la comunidad
  normalización: no | score suficiente
  fragmentos: [('mock_neumonia_2', 'tratamiento', 0.999942), ('mock_neumonia_1', 'otra', 0.997601), ('mock_neumonia_0', 'otra', 0.917704)]
  respuesta: Respuesta simulada con la fuente citada. 

HTA
  normalización: sí | entidad con forma de sigla (score=0.880); normalizada vía SNOMED CT (mock) (score 0.880 -> 0.994)
  fragmentos: [('mock_hipertension_2', 'tratamiento', 0.993937), ('mock_hipertension_1', 'tratamiento', 0.981104)]
  respuesta: Respuesta simulada con la fuente citada. 

amiloidosis
  normalización: sí | score=0.022 < umbral=0.754; normalización SIN match: se mantiene la entidad original; ningún fragmento supera umbral_evidencia=0.554: sin evidencia
  fragmentos: []
  respuesta: No tengo esa información en mis fuentes. | sin evidencia
